# Phase 3 - Cleaning pipeline

Reads the untouched messy tables from `data/raw/` and writes a typed, normalised
copy to `data/clean/`. **The raw layer is never modified.**

Every cleaning decision is stated inline with its rationale. The guiding rule:
*repair what is unambiguously a formatting defect, quarantine what is
unambiguously impossible, and make missingness explicit rather than silently
dropping rows.* Row counts only fall where the brief asks for it (duplicates).

Cleaning decisions are numbered **C1-C12** and referenced from
`docs/FINDINGS.md`.

In [1]:
import os
import re
import sqlite3
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW = os.path.join(ROOT, "data", "raw")
CLEAN = os.path.join(ROOT, "data", "clean")
os.makedirs(CLEAN, exist_ok=True)

raw = {t: pd.read_csv(os.path.join(RAW, t + ".csv"), dtype=str, keep_default_na=False,
                      na_values=[""])
       for t in ["bookings", "customers", "inquiries", "trips", "weather"]}

audit = []          # (step, table, description, rows_before, rows_after)


def log(step, table, desc, before, after):
    audit.append({"step": step, "table": table, "decision": desc,
                  "rows_before": before, "rows_after": after,
                  "delta": after - before})
    print("[%s] %-9s %-58s %6d -> %6d" % (step, table, desc, before, after))


for t, df in raw.items():
    print("%-10s %6d rows x %2d cols" % (t, len(df), df.shape[1]))

bookings     7199 rows x 11 cols
customers    5495 rows x  8 cols
inquiries   10845 rows x  4 cols
trips         922 rows x  5 cols
weather       882 rows x  2 cols


## C1 - Duplicate `booking_id` rows

The brief plants "fully duplicated booking_id rows". Before dropping anything we
check whether any `booking_id` appears more than once with *different* field
values - that would be a genuine conflict needing a survivorship rule, not a
dedupe. If every duplicate id is a byte-identical row, `drop_duplicates()` is
safe and lossless.

In [2]:
b = raw["bookings"]
dupe_ids = b["booking_id"][b["booking_id"].duplicated()].unique()
print("booking_ids appearing more than once: %d" % len(dupe_ids))

conflicting = []
for bid in dupe_ids:
    grp = b[b["booking_id"] == bid].drop_duplicates()
    if len(grp) > 1:
        conflicting.append(bid)
print("of which have CONFLICTING field values: %d" % len(conflicting))

if conflicting:
    print("\n!! conflicting ids need a survivorship rule, not a plain dedupe:")
    print(b[b["booking_id"].isin(conflicting)].sort_values("booking_id").head(20))
else:
    print("-> all duplicates are byte-identical; drop_duplicates() is lossless.")

before = len(b)
b = b.drop_duplicates().reset_index(drop=True)
log("C1", "bookings", "drop fully-duplicated rows (all cols identical)", before, len(b))
assert b["booking_id"].is_unique, "booking_id still not unique after dedupe"

booking_ids appearing more than once: 28
of which have CONFLICTING field values: 0
-> all duplicates are byte-identical; drop_duplicates() is lossless.
[C1] bookings  drop fully-duplicated rows (all cols identical)              7199 ->   7171


## C2 - Mixed date formats

The generator emits three formats: `2024-03-01`, `01/03/2024` (**day-first**),
and `March 1 2024`. Day-first matters: `01/03/2024` is 1 March, not 3 January.
We parse each format explicitly and assert nothing is left unparsed, rather than
letting `pd.to_datetime` guess per-row and silently swap day/month.

In [3]:
def parse_mixed_dates(s):
    """Explicit three-format parse. Returns (parsed_series, per_format_counts)."""
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    counts = {}

    masks = {
        "ISO yyyy-mm-dd": s.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False),
        "dd/mm/yyyy":     s.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False),
        "Month d yyyy":   s.str.match(r"^[A-Za-z]+ \d{1,2} \d{4}$", na=False),
    }
    fmts = {"ISO yyyy-mm-dd": "%Y-%m-%d", "dd/mm/yyyy": "%d/%m/%Y",
            "Month d yyyy": "%B %d %Y"}
    for name, m in masks.items():
        counts[name] = int(m.sum())
        if m.any():
            out.loc[m] = pd.to_datetime(s[m], format=fmts[name])

    unmatched = s.notna() & ~np.logical_or.reduce(list(masks.values()))
    counts["UNPARSED"] = int(unmatched.sum())
    if unmatched.any():
        print("  !! unparsed samples:", s[unmatched].head(5).tolist())
    return out, counts


for tbl, col in [("bookings", "booking_date"), ("customers", "signup_date")]:
    src = b[col] if tbl == "bookings" else raw[tbl][col]
    parsed, counts = parse_mixed_dates(src)
    print("%s.%s -> %s" % (tbl, col, counts))
    assert counts["UNPARSED"] == 0, "unparsed dates in %s.%s" % (tbl, col)
    if tbl == "bookings":
        b["booking_date"] = parsed
    else:
        raw[tbl]["signup_date"] = parsed

# dive_date / inquiry_date / weather_date were generated clean ISO - parse strictly
b["dive_date"] = pd.to_datetime(b["dive_date"], format="%Y-%m-%d")
raw["inquiries"]["inquiry_date"] = pd.to_datetime(raw["inquiries"]["inquiry_date"],
                                                  format="%Y-%m-%d")
raw["weather"]["weather_date"] = pd.to_datetime(raw["weather"]["weather_date"],
                                                format="%Y-%m-%d")
log("C2", "bookings", "parse 3 date formats (day-first honoured)", len(b), len(b))

bookings.booking_date -> {'ISO yyyy-mm-dd': 4317, 'dd/mm/yyyy': 1803, 'Month d yyyy': 1051, 'UNPARSED': 0}
customers.signup_date -> {'ISO yyyy-mm-dd': 3250, 'dd/mm/yyyy': 1394, 'Month d yyyy': 851, 'UNPARSED': 0}
[C2] bookings  parse 3 date formats (day-first honoured)                    7171 ->   7171


## C3 - Booking date sanity

A booking made *after* the dive it books is impossible. We measure lead time and
flag negatives rather than deleting - if any exist they are a systems problem
worth reporting, not noise to hide.

In [4]:
b["lead_time_days"] = (b["dive_date"] - b["booking_date"]).dt.days
bad_lead = b["lead_time_days"] < 0
print("bookings dated after their dive_date: %d" % int(bad_lead.sum()))
print(b["lead_time_days"].describe().round(1).to_string())
log("C3", "bookings", "derive lead_time_days; flag negatives (none expected)",
    len(b), len(b))

bookings dated after their dive_date: 0
count    7171.0
mean       22.4
std        13.3
min         0.0
25%        11.0
50%        23.0
75%        34.0
max        45.0
[C3] bookings  derive lead_time_days; flag negatives (none expected)        7171 ->   7171


## C4 - `status` casing

12 distinct spellings collapse to the 3 canonical values. Strip whitespace,
casefold, then map. Any value that does not map is a hard failure - we would
rather stop than silently bucket an unknown status.

In [5]:
STATUS_MAP = {"completed": "Completed", "cancelled": "Cancelled", "no-show": "No-show"}
print("raw status spellings (%d):" % b["status"].nunique(), sorted(b["status"].unique()))

key = b["status"].str.strip().str.lower()
unmapped = sorted(set(key.dropna()) - set(STATUS_MAP))
assert not unmapped, "unmapped status values: %s" % unmapped
b["status"] = key.map(STATUS_MAP)
print("\n", b["status"].value_counts().to_string())
log("C4", "bookings", "normalise 12 status spellings -> 3 canonical", len(b), len(b))

raw status spellings (12): [' Completed', ' cancelled ', 'CANCELLED', 'COMPLETED', 'Cancelled', 'Completed', 'NO-SHOW', 'No-Show', 'No-show', 'cancelled', 'completed', 'no-show']

 status
Completed    5493
Cancelled    1333
No-show       345
[C4] bookings  normalise 12 status spellings -> 3 canonical                 7171 ->   7171


## C5 - `dive_type` synonyms

14 spellings including the trade abbreviation `DSD` (Discover Scuba Diving) and
`PADI Course`. Mapped to the 3 products named in the brief.

In [6]:
DIVE_TYPE_MAP = {
    "discovery dive": "Discovery Dive", "dsd": "Discovery Dive",
    "discovery": "Discovery Dive",
    "fun dive": "Fun Dive", "fundive": "Fun Dive",
    "course": "Course", "padi course": "Course",
}
print("raw dive_type spellings (%d):" % b["dive_type"].nunique(),
      sorted(b["dive_type"].unique()))

key = b["dive_type"].str.strip().str.lower()
unmapped = sorted(set(key.dropna()) - set(DIVE_TYPE_MAP))
assert not unmapped, "unmapped dive_type values: %s" % unmapped
b["dive_type"] = key.map(DIVE_TYPE_MAP)
print("\n", b["dive_type"].value_counts().to_string())
log("C5", "bookings", "normalise 14 dive_type spellings -> 3 products", len(b), len(b))

raw dive_type spellings (14): ['COURSE', 'Course', 'DISCOVERY DIVE', 'DSD', 'Discovery', 'Discovery Dive', 'FUN DIVE', 'Fun Dive', 'Fun dive', 'PADI Course', 'course', 'discovery dive', 'fun dive', 'fundive']

 dive_type
Fun Dive          3794
Discovery Dive    2158
Course            1219
[C5] bookings  normalise 14 dive_type spellings -> 3 products               7171 ->   7171


## C6 - Price quarantine

Negative prices are impossible and extreme highs are data-entry errors. Two
decisions here, and the second one matters:

1. **Threshold.** The dearest product is a 5-dive Course at roughly Rs 27,500
   list. We quarantine `price_inr <= 0` and `price_inr > 60,000` - the latter is
   over 2x the most expensive thing the shop sells, so nothing legitimate can
   sit above it. Chosen as a business rule, not a percentile, so it stays stable
   as volumes change.
2. **We keep the row.** A booking with a corrupt price is still a real seat on a
   real boat: it must count for `capacity_utilization`, `cancellation_rate` and
   `no_show_rate`. Only its *revenue* is untrustworthy. So we null the price,
   set `price_quarantined`, and every revenue KPI excludes those rows while
   every volume KPI keeps them. Quarantined rows are written to
   `data/clean/quarantine_prices.csv` for audit - nothing is destroyed.

In [7]:
b["price_inr"] = pd.to_numeric(b["price_inr"], errors="coerce")
PRICE_FLOOR, PRICE_CEILING = 0, 60000

q = (b["price_inr"] <= PRICE_FLOOR) | (b["price_inr"] > PRICE_CEILING) | b["price_inr"].isna()
b["price_quarantined"] = q
print("negative / zero      : %d" % int((b["price_inr"] <= 0).sum()))
print("above Rs %d ceiling: %d" % (PRICE_CEILING, int((b["price_inr"] > PRICE_CEILING).sum())))
print("non-numeric          : %d" % int(b["price_inr"].isna().sum()))
print("total quarantined    : %d (%.2f%% of bookings)" % (q.sum(), 100 * q.mean()))
print("\nquarantined values:", sorted(b.loc[q, "price_inr"].dropna().astype(int).tolist()))

b.loc[q, ["booking_id", "customer_id", "dive_date", "dive_type", "price_inr"]] \
 .to_csv(os.path.join(CLEAN, "quarantine_prices.csv"), index=False)
b.loc[q, "price_inr"] = np.nan
print("\nsurviving price_inr distribution:")
print(b["price_inr"].describe().round(0).to_string())
log("C6", "bookings", "quarantine impossible prices (row kept, price nulled)",
    len(b), len(b))

negative / zero      : 15
above Rs 60000 ceiling: 8
non-numeric          : 0
total quarantined    : 23 (0.32% of bookings)

quarantined values: [-26750, -25900, -20650, -14400, -8950, -8000, -7600, -7050, -6550, -5900, -5850, -5650, -5200, -4100, -3300, 161391, 327465, 481124, 501155, 608525, 761763, 1234915, 1605169]

surviving price_inr distribution:
count     7148.0
mean     10007.0
std       7637.0
min       2750.0
25%       4700.0
50%       6800.0
75%      11950.0
max      29700.0
[C6] bookings  quarantine impossible prices (row kept, price nulled)        7171 ->   7171


## C7 - `cancellation_reason`

Two different kinds of null are tangled in this column:

* rows that are **not** cancelled - null is correct and meaningful;
* rows that **are** cancelled but have no reason logged - this is the defect.

We do not impute the second group. Guessing a reason would manufacture the very
number the analysis is meant to measure. They become an explicit `"Unknown"`
category, because *how much of the cancellation book is unexplained* is itself a
finding.

In [8]:
canc = b["status"] == "Cancelled"
missing_reason = canc & b["cancellation_reason"].isna()
print("cancelled bookings              : %d" % int(canc.sum()))
print("  ...with no reason logged      : %d (%.1f%% of cancellations)"
      % (missing_reason.sum(), 100 * missing_reason.sum() / canc.sum()))
print("non-cancelled rows with a reason: %d  (should be 0)"
      % int((~canc & b["cancellation_reason"].notna()).sum()))

b.loc[missing_reason, "cancellation_reason"] = "Unknown"
b.loc[~canc, "cancellation_reason"] = pd.NA
print("\n", b.loc[canc, "cancellation_reason"].value_counts().to_string())
log("C7", "bookings", "cancelled+null reason -> 'Unknown' (not imputed)", len(b), len(b))

cancelled bookings              : 1333
  ...with no reason logged      : 196 (14.7% of cancellations)
non-cancelled rows with a reason: 0  (should be 0)

 cancellation_reason
Weather     538
Customer    442
Unknown     196
Medical     157
[C7] bookings  cancelled+null reason -> 'Unknown' (not imputed)             7171 ->   7171


In [9]:
b["num_dives"] = pd.to_numeric(b["num_dives"], errors="raise").astype(int)
b["boat_slot"] = pd.to_numeric(b["boat_slot"], errors="raise").astype(int)
print(b[["num_dives", "boat_slot"]].describe().round(2).to_string())

       num_dives  boat_slot
count    7171.00    7171.00
mean        2.25       5.59
std         1.37       3.03
min         1.00       1.00
25%         1.00       3.00
50%         2.00       6.00
75%         4.00       8.00
max         5.00      12.00

## C8 - `nationality`

54 raw strings for 12 real countries: casing, ISO-2 codes, demonyms
(`Indian`), constituent-country names (`England`) and informal names
(`Holland`). Mapped explicitly - no fuzzy matching, because a wrong silent
merge here would corrupt any nationality cut of the data.

In [10]:
c = raw["customers"]
NATIONALITY_MAP = {}
for canon, variants in {
    "India": ["india", "in", "indian"],
    "United Kingdom": ["united kingdom", "uk", "u.k.", "british", "england"],
    "Germany": ["germany", "de", "german"],
    "Russia": ["russia", "ru", "russian"],
    "Israel": ["israel", "il", "israeli"],
    "France": ["france", "fr", "french"],
    "United States": ["united states", "usa", "us", "u.s.a.", "american"],
    "Australia": ["australia", "au", "australian"],
    "Italy": ["italy", "it", "italian"],
    "South Korea": ["south korea", "korea", "kr", "korean"],
    "Japan": ["japan", "jp", "japanese"],
    "Netherlands": ["netherlands", "nl", "dutch", "holland"],
}.items():
    for v in variants:
        NATIONALITY_MAP[v] = canon

print("raw nationality strings: %d" % c["nationality"].nunique())
key = c["nationality"].str.strip().str.lower()
unmapped = sorted(set(key.dropna()) - set(NATIONALITY_MAP))
assert not unmapped, "unmapped nationality values: %s" % unmapped
c["nationality"] = key.map(NATIONALITY_MAP)
print("clean nationalities   : %d\n" % c["nationality"].nunique())
print(c["nationality"].value_counts().to_string())
log("C8", "customers", "54 nationality strings -> 12 countries", len(c), len(c))

raw nationality strings: 54


clean nationalities   : 12

nationality
India             3028
United Kingdom     395
Germany            314
Russia             289
France             231
United States      224
Netherlands        215
Israel             214
Italy              165
Australia          153
South Korea        150
Japan              117
[C8] customers 54 nationality strings -> 12 countries                       5495 ->   5495


## C9 - `cert_level`

Casing plus stray internal double-spaces and leading/trailing whitespace.
Collapse whitespace, then map on a casefolded key.

In [11]:
CERT_ORDER = ["None", "Open Water", "Advanced", "Rescue", "Divemaster"]
CERT_MAP = {v.lower(): v for v in CERT_ORDER}
print("raw cert_level strings: %d" % c["cert_level"].nunique())

key = c["cert_level"].str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
unmapped = sorted(set(key.dropna()) - set(CERT_MAP))
assert not unmapped, "unmapped cert_level values: %s" % unmapped
c["cert_level"] = pd.Categorical(key.map(CERT_MAP), categories=CERT_ORDER, ordered=True)
print("clean cert_levels     : %d\n" % c["cert_level"].nunique())
print(c["cert_level"].value_counts().reindex(CERT_ORDER).to_string())
log("C9", "customers", "26 cert_level strings -> 5 ordered levels", len(c), len(c))

raw cert_level strings: 26
clean cert_levels     : 5

cert_level
None          2602
Open Water    1735
Advanced       822
Rescue         227
Divemaster     109
[C9] customers 26 cert_level strings -> 5 ordered levels                    5495 ->   5495


## C10 - `age` outliers and nulls

Recreational diving has a hard lower bound: PADI certifies from age 10, and no
operator puts a 3-year-old on a boat. Ages outside **10-90** are impossible
rather than merely unusual, so they are nulled (not clipped - clipping would
invent a plausible age and pull the mean). Genuinely missing ages stay missing;
we do not impute, because age is used only descriptively.

In [12]:
c["age"] = pd.to_numeric(c["age"], errors="coerce")
AGE_MIN, AGE_MAX = 10, 90
impossible = c["age"].notna() & ((c["age"] < AGE_MIN) | (c["age"] > AGE_MAX))
print("missing age at load : %d (%.1f%%)" % (c["age"].isna().sum(),
                                             100 * c["age"].isna().mean()))
print("impossible ages     : %d -> %s" % (impossible.sum(),
                                          sorted(c.loc[impossible, "age"].astype(int).unique())))
c.loc[impossible, "age"] = np.nan
c["age_missing"] = c["age"].isna()
print("missing after clean : %d (%.1f%%)" % (c["age"].isna().sum(),
                                             100 * c["age"].isna().mean()))
print("\n", c["age"].describe().round(1).to_string())
log("C10", "customers", "null ages outside 10-90 (not clipped)", len(c), len(c))

missing age at load : 397 (7.2%)
impossible ages     : 35 -> [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(115), np.int64(118), np.int64(121)]
missing after clean : 432 (7.9%)

 count    5063.0
mean       33.5
std         9.7
min        12.0
25%        27.0
50%        34.0
75%        40.0
max        74.0
[C10] customers null ages outside 10-90 (not clipped)                        5495 ->   5495


## C11 - `acquisition_channel` nulls

Missing channel becomes an explicit `"Unknown"` rather than being dropped. The
share of revenue that cannot be attributed to any channel is a finding in its
own right - dropping these rows would quietly flatter every channel's numbers.

In [13]:
print("null acquisition_channel: %d (%.1f%%)" % (c["acquisition_channel"].isna().sum(),
                                                 100 * c["acquisition_channel"].isna().mean()))
c["acquisition_channel"] = c["acquisition_channel"].fillna("Unknown")
print("\n", c["acquisition_channel"].value_counts().to_string())
log("C11", "customers", "null acquisition_channel -> 'Unknown' (kept, not dropped)",
    len(c), len(c))

null acquisition_channel: 497 (9.0%)

 acquisition_channel
Walk-in          1144
OTA              1118
Website           958
Referral          662
Instagram         599
Hotel Partner     517
Unknown           497
[C11] customers null acquisition_channel -> 'Unknown' (kept, not dropped)    5495 ->   5495


## C12 - Referential integrity

Check every join the KPIs depend on before we rely on it.

In [14]:
t = raw["trips"].copy()
w = raw["weather"].copy()
i = raw["inquiries"].copy()
t["boat_capacity"] = pd.to_numeric(t["boat_capacity"], errors="raise").astype(int)

checks = {
    "bookings.customer_id -> customers": (~b["customer_id"].isin(c["customer_id"])).sum(),
    "bookings.trip_id -> trips": (~b["trip_id"].isin(t["trip_id"])).sum(),
    "bookings.dive_date -> weather": (~b["dive_date"].isin(w["weather_date"])).sum(),
    "inquiries.converted_booking_id -> bookings":
        (~i["converted_booking_id"].dropna().isin(b["booking_id"])).sum(),
    "customers with no booking": (~c["customer_id"].isin(b["customer_id"])).sum(),
    "trips with no booking": (~t["trip_id"].isin(b["trip_id"])).sum(),
}
for k, v in checks.items():
    print("%-46s %d orphan(s)" % (k, v))

over = (b.groupby("trip_id").size().rename("seats")
        .to_frame().join(t.set_index("trip_id")["boat_capacity"]))
over["overbooked"] = over["seats"] > over["boat_capacity"]
print("\ntrips booked beyond boat_capacity: %d" % int(over["overbooked"].sum()))
log("C12", "all", "referential integrity + capacity checks", len(b), len(b))

bookings.customer_id -> customers              0 orphan(s)


bookings.trip_id -> trips                      0 orphan(s)
bookings.dive_date -> weather                  0 orphan(s)
inquiries.converted_booking_id -> bookings     0 orphan(s)
customers with no booking                      0 orphan(s)
trips with no booking                          0 orphan(s)

trips booked beyond boat_capacity: 0
[C12] all       referential integrity + capacity checks                      7171 ->   7171


## Derived analysis columns

Two different notions of "season" are needed and they must not be confused:

* **`season`** - Peak / Shoulder / Monsoon-Closed, the within-year trading
  period, carried on `trips` per the brief.
* **`season_year`** - the 2.5 chronological seasons (2023-24 / 2024-25 /
  2025-26) used for season-over-season trend. 2023-24 is a **half** season
  (Jan-May 2024 only) so its volumes are not comparable to the two full ones;
  trend claims are made on the two full seasons.

In [15]:
def season_year(d):
    if d <= pd.Timestamp("2024-05-31"):
        return "2023-24 (half)"
    if d <= pd.Timestamp("2025-05-31"):
        return "2024-25"
    return "2025-26"


b["season_year"] = b["dive_date"].map(season_year)
b["dive_month"] = b["dive_date"].dt.month
b["dive_ym"] = b["dive_date"].dt.to_period("M").astype(str)
b["is_weekend"] = b["dive_date"].dt.weekday >= 5
b["revenue_inr"] = np.where(b["status"] == "Completed", b["price_inr"], np.nan)
i["season_year"] = i["inquiry_date"].map(season_year)
i["converted"] = i["converted_booking_id"].notna()

print(b.groupby("season_year").agg(
    bookings=("booking_id", "size"),
    first_dive=("dive_date", "min"),
    last_dive=("dive_date", "max")).to_string())

                bookings first_dive  last_dive
season_year                                   
2023-24 (half)      1891 2024-01-01 2024-05-31
2024-25             2884 2024-06-01 2025-05-31
2025-26             2396 2025-06-03 2026-05-31


## Write `data/clean/`

CSVs plus a typed SQLite database. Unlike the raw DB, the clean one declares
**primary keys and foreign keys** - the constraints are what make it a trusted
layer, and they now hold because C1-C12 removed every violation.

In [16]:
clean = {"bookings": b, "customers": c, "inquiries": i, "trips": t, "weather": w}
for name, df in clean.items():
    df.to_csv(os.path.join(CLEAN, name + ".csv"), index=False)
    print("%-10s %6d rows -> data/clean/%s.csv" % (name, len(df), name))

DDL = """
DROP TABLE IF EXISTS bookings; DROP TABLE IF EXISTS inquiries;
DROP TABLE IF EXISTS customers; DROP TABLE IF EXISTS trips;
DROP TABLE IF EXISTS weather;

CREATE TABLE customers (
  customer_id TEXT PRIMARY KEY, first_name TEXT, last_name TEXT,
  nationality TEXT NOT NULL, age REAL, age_missing INTEGER,
  cert_level TEXT NOT NULL, acquisition_channel TEXT NOT NULL, signup_date TEXT);

CREATE TABLE trips (
  trip_id TEXT PRIMARY KEY, dive_site TEXT NOT NULL, difficulty TEXT NOT NULL,
  boat_capacity INTEGER NOT NULL, season TEXT NOT NULL);

CREATE TABLE weather (weather_date TEXT PRIMARY KEY, sea_condition TEXT NOT NULL);

CREATE TABLE bookings (
  booking_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL, booking_date TEXT,
  dive_date TEXT NOT NULL, dive_type TEXT NOT NULL, num_dives INTEGER NOT NULL,
  price_inr REAL, status TEXT NOT NULL, cancellation_reason TEXT,
  trip_id TEXT NOT NULL, boat_slot INTEGER, price_quarantined INTEGER,
  lead_time_days INTEGER, season_year TEXT, dive_month INTEGER, dive_ym TEXT,
  is_weekend INTEGER, revenue_inr REAL,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
  FOREIGN KEY (trip_id) REFERENCES trips(trip_id),
  FOREIGN KEY (dive_date) REFERENCES weather(weather_date));

CREATE TABLE inquiries (
  inquiry_id TEXT PRIMARY KEY, inquiry_date TEXT NOT NULL, channel TEXT NOT NULL,
  converted_booking_id TEXT, season_year TEXT, converted INTEGER,
  FOREIGN KEY (converted_booking_id) REFERENCES bookings(booking_id));

CREATE INDEX idx_bk_trip ON bookings(trip_id);
CREATE INDEX idx_bk_cust ON bookings(customer_id);
CREATE INDEX idx_bk_date ON bookings(dive_date);
"""

DB = os.path.join(CLEAN, "dive_ops_clean.db")
if os.path.exists(DB):
    os.remove(DB)
con = sqlite3.connect(DB)
con.executescript(DDL)
for name in ["customers", "trips", "weather", "bookings", "inquiries"]:
    df = clean[name].copy()
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime("%Y-%m-%d")
        elif isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].astype(str)
    df.to_sql(name, con, if_exists="append", index=False)

con.execute("PRAGMA foreign_keys = ON")
violations = con.execute("PRAGMA foreign_key_check").fetchall()
print("\nforeign_key_check violations: %d" % len(violations))
assert not violations, violations
for name in clean:
    print("  %-10s %6d rows in SQLite" % (name, con.execute(
        "SELECT COUNT(*) FROM %s" % name).fetchone()[0]))
con.commit()
con.close()

bookings     7171 rows -> data/clean/bookings.csv
customers    5495 rows -> data/clean/customers.csv
inquiries   10845 rows -> data/clean/inquiries.csv
trips         922 rows -> data/clean/trips.csv
weather       882 rows -> data/clean/weather.csv



foreign_key_check violations: 0
  bookings     7171 rows in SQLite
  customers    5495 rows in SQLite
  inquiries   10845 rows in SQLite
  trips         922 rows in SQLite
  weather       882 rows in SQLite


## Cleaning audit trail

In [17]:
adf = pd.DataFrame(audit)
adf.to_csv(os.path.join(CLEAN, "cleaning_audit.csv"), index=False)
print(adf.to_string(index=False))
print("\nBookings: %d raw rows -> %d clean rows (%d duplicates removed, 0 rows dropped for any other reason)"
      % (len(raw["bookings"]), len(b), len(raw["bookings"]) - len(b)))

step     table                                                  decision  rows_before  rows_after  delta
  C1  bookings           drop fully-duplicated rows (all cols identical)         7199        7171    -28
  C2  bookings                 parse 3 date formats (day-first honoured)         7171        7171      0
  C3  bookings     derive lead_time_days; flag negatives (none expected)         7171        7171      0
  C4  bookings              normalise 12 status spellings -> 3 canonical         7171        7171      0
  C5  bookings            normalise 14 dive_type spellings -> 3 products         7171        7171      0
  C6  bookings     quarantine impossible prices (row kept, price nulled)         7171        7171      0
  C7  bookings          cancelled+null reason -> 'Unknown' (not imputed)         7171        7171      0
  C8 customers                    54 nationality strings -> 12 countries         5495        5495      0
  C9 customers                 26 cert_level strings ->